# S6E10 — 18 RealMLP-rich "recipe mix": seeds that differ in more than the seed

> **Pipeline position:** 10 RealMLP-rich, 12 / 17 (recipe v2, variant A+B) → **18 recipe mix** → 11 stack

| | |
|---|---|
| **Input** | `X_rich_*` (09), `X_teach_*` (16), `y_train.parquet`; OOF/test files of the stack members |
| **Output** | `preds/realmlp_rich_mix_{ref,v2,ab}_folds/` (checkpoints), `oof_/test_realmlp_rich_mix_{ref,v2,ab}.npy`, averaged `oof_/test_realmlp_rich_mix.npy` |
| **Runtime** | ≈ 3 min per fold and variant with 8 networks on an RTX 3070 Ti → ≈ 45 min |
| **Result** | see *Results & decisions* |

**Why:** none of the variants of `12`–`17` beat RealMLP-rich on its own, but two of them **rank rows differently**: the v2 recipe (Spearman 0.993 with the reference vs 0.996 between plain seeds) and variant A+B of `17`. On fold 0, averaging a plain second seed into RealMLP-rich gains +0.00002, averaging v2 gains +0.00005–0.00008. So more *seeds* are worth more if they are *different recipes*. This notebook trains three such members on all folds and averages them into one stack member, `realmlp_rich_mix`:

| Member | Input | Recipe | Seed |
|---|---|---|---|
| `ref` | `X_rich` (as `10`) | `10` | 3 (seeds 0–2 already exist in `10`) |
| `v2` | `X_rich` | v2 of `12` (`flat_anneal`, wd 0.015, `invsqrtp1e-3`, 3 epochs) | 0 |
| `ab` | `X_rich` + teachers + delay twins + 2 TE keys (variant A+B of `17`) | `10` | 0 |

All with **8 networks** (the source-notebook setting; affordable on the GPU).

**Acceptance:** nested CV of the stack exactly as in `11` (LR C = 10 on logits, fitted on 4 folds of OOF, scored on the 5th), two configurations: (1) `realmlp_rich_mix` **added** as an 11th member; (2) `realmlp_rich_mix` **replacing** `realmlp_rich`. Bar: ≥ +0.00003 over the current 0.96178 and positive on ≥ 4 of 5 folds. `11` itself is only changed after acceptance.

> **Glossary:** *Recipe mix* = averaging models that differ in training settings or inputs, not only in the random seed. *Nested CV* = the stacker is scored only on rows it was not fitted on.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_ENS = 8
RUN_TRAIN = True     # False: only load the checkpoints and evaluate
PROC_DIR = next(p for p in [Path('data/processed'), Path('../data/processed')] if (p / 'X_teach_train.parquet').exists())
PREDS_DIR = next(p for p in [Path('preds'), Path('../preds')] if p.exists())
X0 = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X0_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
TCH = pd.read_parquet(PROC_DIR / 'X_teach_train.parquet'); TCH_test = pd.read_parquet(PROC_DIR / 'X_teach_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
# The project-wide split: identical folds in every notebook keep OOF predictions stackable.
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X0, y))
print(DEVICE, '| networks:', N_ENS, X0.shape)

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 05:04:36.096000 15316 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


cuda | networks: 8 (699635, 92)


## 1. Inputs and recipes (code of `10`, `12`, `17`)

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
BASE_TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
BASE_TE = ['flight_distance', 'age', 'fd_div10', 'fd_mod1000', 'key_age_class', 'key_fd_class', 'key_age_travel']

def build(ab):
    X, Xt = X0.copy(), X0_test.copy()
    twins, te_src = dict(BASE_TWINS), list(BASE_TE)
    if ab:      # variant A+B of 17
        X = pd.concat([X, TCH], axis=1); Xt = pd.concat([Xt, TCH_test], axis=1)
        for D in (X, Xt):
            D['arr_delay_int'] = (D['departure_delay_in_minutes'] - D['delay_recovered']).round().astype(int)
            code_ = lambda c: D[c].cat.codes.astype(int) if hasattr(D[c], 'cat') else D[c].astype('category').cat.codes.astype(int)
            D['key_cls_trv_gen'] = code_('class') * 100 + code_('type_of_travel') * 10 + code_('gender')
            D['key_fd_time'] = D['flight_distance'].astype(int) * 10 + D['departure/arrival_time_convenient'].astype(int)
        twins.update({'dep_delay_cat': 'departure_delay_in_minutes', 'arr_delay_cat': 'arr_delay_int'})
        te_src += ['key_cls_trv_gen', 'key_fd_time']
    return X, Xt, twins, te_src

PARAMS_REF = dict(n_epochs=4, n_ens=N_ENS, batch_size=256, use_early_stopping=False, val_metric_name='1-auc_ovr', device=DEVICE, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'])
PARAMS_V2 = {**PARAMS_REF, 'lr_sched': 'flat_anneal', 'wd': 0.015, 'p_drop_sched': 'invsqrtp1e-3', 'n_epochs': 3}
MEMBERS = {'ref': dict(ab=False, params=PARAMS_REF, seed=3), 'v2': dict(ab=False, params=PARAMS_V2, seed=0), 'ab': dict(ab=True, params=PARAMS_REF, seed=0)}

def make_runner(ab, params, seed):
    X, Xt, twins, te_src = build(ab)
    FULL = pd.concat([X, Xt], ignore_index=True)
    levels = {t: sorted(FULL[s].astype(int).unique()) for t, s in twins.items()}
    keys = [c for c in X.columns if c.startswith('key_')]
    drop = ['orig_prob'] + keys + (['arr_delay_int'] if ab else [])
    cat_cols = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(twins)
    def nn_frame(D):
        D = D.copy()
        for t, s in twins.items(): D[t] = pd.Categorical(D[s].astype(int), categories=levels[t])
        D = D.drop(columns=drop)
        assert not D.isna().any().any()
        return D
    # In-fold target encoding: fitted on the fold's TRAINING part only (inner cv=5), applied to validation/test.
    def add_te(Xtr, ytr, others):
        te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
        cols = [f'te_{c}' for c in te_src]
        tr_te = pd.DataFrame(te.fit_transform(Xtr[te_src].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
        outs = [pd.DataFrame(te.transform(o[te_src].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
        return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]
    def fit_predict(tr_idx, va_idx):
        Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], Xt])
        Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
        model = RealMLP_TD_Classifier(**{**params, 'random_state': seed})
        # X_val only for logging: early stopping is off, so the OOF rows never choose anything.
        model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=cat_cols)
        out = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
        del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == 'cuda' else None
        return out
    return fit_predict

## 2. Train all folds of the three members (resumable)
Every fold is checkpointed in `preds/realmlp_rich_mix_<member>_folds/`; a rerun continues where it stopped.

In [3]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
sig = lambda z: 1 / (1 + np.exp(-z))
OOFM, TESTM = {}, {}
for name, cfg in MEMBERS.items():
    d = PREDS_DIR / f'realmlp_rich_mix_{name}_folds'; d.mkdir(exist_ok=True)
    runner = None; oof = np.full(len(y), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        fo, ft = d / f'oof_{k}.npy', d / f'test_{k}.npy'
        if not (fo.exists() and ft.exists()):
            if not RUN_TRAIN: raise SystemExit(f'missing checkpoint {fo}')
            runner = runner or make_runner(cfg['ab'], cfg['params'], cfg['seed'])
            t0 = time.time(); p_va, p_te = runner(tr, va); np.save(fo, p_va); np.save(ft, p_te)
            print(f'{name} fold {k}: AUC {roc_auc_score(y[va], p_va):.5f} ({(time.time()-t0)/60:.1f} min)', flush=True)
        oof[va] = np.load(fo); tests.append(np.load(ft))
    OOFM[name], TESTM[name] = oof, np.mean(tests, axis=0)
    np.save(PREDS_DIR / f'oof_realmlp_rich_mix_{name}.npy', oof); np.save(PREDS_DIR / f'test_realmlp_rich_mix_{name}.npy', TESTM[name])
    print(f'{name}: OOF AUC {roc_auc_score(y, oof):.5f} | per fold', [round(roc_auc_score(y[va], oof[va]), 5) for _, va in FOLDS], flush=True)

# The mix = mean of the three members' logits (as the seed averages of 10).
o_mix = np.mean([lg(OOFM[n]) for n in MEMBERS], axis=0); t_mix = np.mean([lg(TESTM[n]) for n in MEMBERS], axis=0)
np.save(PREDS_DIR / 'oof_realmlp_rich_mix.npy', sig(o_mix)); np.save(PREDS_DIR / 'test_realmlp_rich_mix.npy', sig(t_mix))
old = lg(np.load(PREDS_DIR / 'oof_realmlp_rich_avg.npy'))
print(f'\nrealmlp_rich_mix OOF AUC {roc_auc_score(y, o_mix):.5f} | realmlp_rich_avg (3 seeds of 10) {roc_auc_score(y, old):.5f}')
print('Spearman between members:', {f'{a}-{b}': round(spearmanr(lg(OOFM[a]), lg(OOFM[b]))[0], 4) for a, b in [('ref', 'v2'), ('ref', 'ab'), ('v2', 'ab')]},
      '| mix vs realmlp_rich_avg', round(spearmanr(o_mix, old)[0], 4))

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


ab fold 4: AUC 0.96126 (1.6 min)


ab: OOF AUC 0.96131 | per fold [0.96149, 0.96031, 0.96207, 0.96147, 0.96126]



realmlp_rich_mix OOF AUC 0.96134 | realmlp_rich_avg (3 seeds of 10) 0.96129


Spearman between members: {'ref-v2': np.float64(0.9944), 'ref-ab': np.float64(0.9964), 'v2-ab': np.float64(0.993)} | mix vs realmlp_rich_avg 0.9983


## 3. Nested CV of the stack (exactly the LR procedure of `11`)

In [4]:
as_logit = lambda o: lg(o) if (o.min() >= 0 and o.max() <= 1) else o
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp_avg',
         'realmlp_rich': 'realmlp_rich_avg', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
def load(names):
    O = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float)) for s in names.values()])
    T = np.column_stack([as_logit(np.load(PREDS_DIR / f'test_{s}.npy').astype(float)) for s in names.values()])
    return O, T
def nested(O):
    r, W = [], []
    for tr, va in FOLDS:
        w = LogisticRegression(C=10.0, max_iter=500).fit(O[tr], y[tr]).coef_[0]; W.append(w); r.append(roc_auc_score(y[va], O[va] @ w))
    return np.array(r), np.mean(W, axis=0)
CONFIGS = {'current 10': NAMES,
           '+ mix (11 members)': {**NAMES, 'realmlp_rich_mix': 'realmlp_rich_mix'},
           'mix replaces realmlp_rich': {**{k: v for k, v in NAMES.items() if k != 'realmlp_rich'}, 'realmlp_rich_mix': 'realmlp_rich_mix'}}
R, WTS = {}, {}
for cname, names in CONFIGS.items():
    O, _ = load(names); R[cname], WTS[cname] = nested(O)
tab = pd.DataFrame({c: list(r) + [r.mean()] for c, r in R.items()}, index=[f'fold {i}' for i in range(5)] + ['mean']).T
print(tab.round(5).to_string())
for c in list(CONFIGS)[1:]:
    d = R[c] - R['current 10']
    print(f'{c:28s} gain {d.mean():+.5f} | positive folds {(d > 0).sum()}/5 | per fold {np.round(d, 5).tolist()}')
for c in list(CONFIGS)[1:]:
    print(f'\nweights ({c}):', dict(zip(CONFIGS[c], WTS[c].round(3))))

                            fold 0   fold 1   fold 2   fold 3   fold 4     mean
current 10                 0.96191  0.96095  0.96240  0.96194  0.96170  0.96178
+ mix (11 members)         0.96194  0.96094  0.96241  0.96195  0.96171  0.96179
mix replaces realmlp_rich  0.96195  0.96093  0.96241  0.96198  0.96172  0.96180
+ mix (11 members)           gain +0.00001 | positive folds 4/5 | per fold [3e-05, -1e-05, 1e-05, 1e-05, 1e-05]
mix replaces realmlp_rich    gain +0.00002 | positive folds 4/5 | per fold [4e-05, -2e-05, 1e-05, 4e-05, 2e-05]

weights (+ mix (11 members)): {'cat_v1': np.float64(-0.036), 'cat_v2': np.float64(-0.091), 'xgb': np.float64(0.047), 'lgbm_tuned': np.float64(0.169), 'mlp': np.float64(-0.017), 'realmlp': np.float64(-0.16), 'realmlp_rich': np.float64(0.149), 'tabpfn': np.float64(0.628), 'tabpfn_raw': np.float64(0.132), 'tabicl': np.float64(-0.089), 'realmlp_rich_mix': np.float64(0.275)}

weights (mix replaces realmlp_rich): {'cat_v1': np.float64(-0.022), 'cat_v2': np.

## 4. Test-vs-OOF consistency of the new member
Mean and std of the logits, OOF vs test (they should agree), and Spearman between pairs of members on OOF vs on test.

In [5]:
o_l, t_l = lg(np.load(PREDS_DIR / 'oof_realmlp_rich_mix.npy')), lg(np.load(PREDS_DIR / 'test_realmlp_rich_mix.npy'))
print(f'logits OOF mean {o_l.mean():.4f} std {o_l.std():.4f} | test mean {t_l.mean():.4f} std {t_l.std():.4f}')
for other in ['realmlp_rich_avg', 'tabpfn_avg', 'lgbm_tuned_final']:
    oo, tt = as_logit(np.load(PREDS_DIR / f'oof_{other}.npy').astype(float)), as_logit(np.load(PREDS_DIR / f'test_{other}.npy').astype(float))
    print(f'Spearman mix vs {other:18s} OOF {spearmanr(o_l, oo)[0]:.4f} | test {spearmanr(t_l, tt)[0]:.4f}')

logits OOF mean -0.4208 std 3.0462 | test mean -0.4220 std 3.0377


Spearman mix vs realmlp_rich_avg   OOF 0.9983 | test 0.9992


Spearman mix vs tabpfn_avg         OOF 0.9793 | test 0.9812


Spearman mix vs lgbm_tuned_final   OOF 0.9809 | test 0.9841


## Results & decisions

**Members (8 networks each, all 5 folds, RTX 3070 Ti, ≈ 1.1–1.6 min per fold):**

| Member | OOF AUC | Fold AUCs |
|---|---|---|
| `ref` (recipe of 10, seed 3) | 0.96124 | 0.96143 / 0.96022 / 0.96202 / 0.96135 / 0.96119 |
| `v2` (recipe v2 of 12) | 0.96121 | 0.96145 / 0.96025 / 0.96192 / 0.96136 / 0.96110 |
| `ab` (variant A+B of 17) | 0.96131 | 0.96149 / 0.96031 / 0.96207 / 0.96147 / 0.96126 |
| **`realmlp_rich_mix` (logit mean)** | **0.96134** | Spearman with `realmlp_rich_avg` 0.998 |
| *`realmlp_rich_avg` (3 seeds of 10, 4 networks)* | *0.96129* | |

**Nested CV of the stack (LR C = 10, as in `11`; "current 10" reproduces 0.96178):**

| Configuration | Nested CV | Gain | Positive folds |
|---|---|---|---|
| current 10 | 0.96178 | – | – |
| + mix as 11th member | 0.96179 | +0.00001 | 4/5 |
| mix replaces `realmlp_rich` | 0.96180 | +0.00002 | 4/5 |
| all six RealMLP-rich runs averaged, replacing `realmlp_rich` (one extra check, no training) | 0.96179 | +0.00001 | 4/5 |

**Decision: rejected (below the +0.00003 bar), step closed; the stack stays at 10 members.** The mix is a slightly better RealMLP-rich (+0.00005 OOF), and the stack prefers it (weight 0.435 vs 0.39), but the gain to the stack is +0.00002 — the size of the LB noise divided by twenty. Test-vs-OOF consistency of the mix is fine (logit mean −0.421 vs −0.422, std 3.046 vs 3.038), so it could serve as a variant member if ever needed.

**Takeaway:** recipe diversity does rank rows differently (Spearman 0.993–0.996 between members vs 0.9955 between plain seeds), but after averaging, what reaches the stack is almost the same model as the seed average it replaces.


## Glossary
* **Recipe mix** — an average of models that differ in training recipe or inputs, not only in the seed.
* **Nested CV** — the stacker is fitted on OOF rows of 4 folds and scored on the 5th.
* **Test-vs-OOF consistency** — the same model's test predictions should look statistically like its OOF predictions.